In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf

import threading
from threading import RLock

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from indoNLP.preprocessing import replace_slang
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
import torch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import spacy

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from tqdm.notebook import tqdm
import re
from datetime import datetime

In [ ]:
from config import config

## Load Data Saham

## Load Data IHSG

# Data Loading

# Data Preprocessing

## Handle Missing Value

In [ ]:
missing_value = df.isnull().sum()
missing_value[missing_value > 0]

In [ ]:
missing_percentage = (missing_value / len(df)) * 100
missing_data = pd.DataFrame({
    'Missing Values': missing_value,
    'Percentage': missing_percentage
}).sort_values(by='Missing Values', ascending=False)

missing_data[missing_data['Missing Values'] > 0]

In [ ]:
missing_value = news.isnull().sum()
missing_value[missing_value > 0]

## News Data Pre-Processing

### Data Standarization

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    
    text = text.lower()
    pola_persen = r'(\d+),(\d+)\s*%'
    def ubah_ke_teks_angka(match):
        depan = match.group(1)
        belakang = match.group(2)
        return f"{depan} koma {belakang} persen"
    
    pola_rupiah = r'rp\.?\s*([\d\.]+)'
    def format_rupiah(match):
        angka_bersih = match.group(1).replace('.', '')
        return f"{angka_bersih} rupiah"
    text = re.sub(pola_rupiah, format_rupiah, text)
    text = re.sub(pola_persen, ubah_ke_teks_angka, text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\bcom\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bdot\b', '.', text, flags=re.IGNORECASE)
    for word in media_keywords:
        text = re.sub(r'\b' + re.escape(word) + r'\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()    

In [ ]:
with ThreadPoolExecutor() as executor:
    futures_title = [executor.submit(clean_text, title) for title in news['title']]
    futures_content = [executor.submit(clean_text, content) for content in news['content']]
    
    news['title_clean'] = [f.result() for f in futures_title]
    news['content_clean'] = [f.result() for f in futures_content]

news

### Abbreviation Processing

In [ ]:
def abbreviation_processing(text):
    return replace_slang(text)

In [ ]:
with ThreadPoolExecutor() as executor:
    futures_title = [executor.submit(abbreviation_processing, title) for title in news['title_clean']]
    futures_content = [executor.submit(abbreviation_processing, content) for content in news['content_clean']]
    
    news['title_abbreviation'] = [f.result() for f in futures_title]
    news['content_abbreviation'] = [f.result() for f in futures_content]

news

### Named Entity Recognition

In [ ]:
thread_local = threading.local()
def get_nlp():
    if not hasattr(thread_local, "nlp"):
        thread_local.nlp = spacy.load("id_ner_spacy_indonesian")
    return thread_local.nlp

def pipeline_ner(text):
    if not isinstance(text, str) or not text.strip():
        return [] 
    nlp = get_nlp()
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

In [ ]:
entities_title = []
entities_content = []
with ThreadPoolExecutor() as executor:
    futures_title = [executor.submit(pipeline_ner, title) for title in news['title_abbreviation']]
    futures_content = [executor.submit(pipeline_ner, content) for content in news['content_abbreviation']]
    
    news['title_entity'] = [f.result() for f in futures_title]
    news['content_entity'] = [f.result() for f in futures_content]

news

### Stopword Handling

In [ ]:
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

def handle_stopword(text):
    return stopword_remover.remove(text) 

In [ ]:
with ThreadPoolExecutor() as executor:
    futures_title = [executor.submit(handle_stopword, title) for title in news['title_abbreviation']]
    futures_content = [executor.submit(handle_stopword, content) for content in news['content_abbreviation']]
    
    news['title_stopword_removed'] = [f.result() for f in futures_title]
    news['content_stopword_removed'] = [f.result() for f in futures_content]

news

### Stemming Word

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer(True)

def stem_word(text):
    return stemmer.stem(text)

In [ ]:
with ThreadPoolExecutor() as executor:
    futures_title = [executor.submit(stem_word, title) for title in news['title_stopword_removed']]
    futures_content = [executor.submit(stem_word, content) for content in news['content_stopword_removed']]
    
    news['title_stemmed'] = [f.result() for f in futures_title]
    news['content_stemmed'] = [f.result() for f in futures_content]

news

### Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

def prepare_and_tokenize(news_item, tokenizer, max_length=512):
    news_item['full_text'] = news_item['title_stemmed'].fillna('').astype(str) + " " + news_item['content_stemmed'].fillna('').astype(str)
    news_item['full_text'] = news_item['full_text'].str.strip()

    news_item.loc[news_item['full_text'] == '', 'full_text'] = tokenizer.unk_token
    
    encoding = tokenizer(
        news_item['full_text'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    news_item['input_ids'] = encoding['input_ids'].squeeze().tolist()
    news_item['attention_mask'] = encoding['attention_mask'].squeeze().tolist()

    return news_item

In [ ]:
news = prepare_and_tokenize(news, tokenizer)

news

### Sentimen Analysis

In [ ]:
model_name = 'agufsamudra/indo-sentiment-analysis'
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

def get_sentiment(input_ids, attention_mask):
    if not input_ids or len(input_ids) == 0:
        return 'UNKNOWN', 0.0
    
    input_ids_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    attention_mask_tensor = torch.tensor([attention_mask], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
    
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)
    pred_id = torch.argmax(probs, dim=-1).item()
    score = probs[0][pred_id].item()

    label = 'POSITIVE' if pred_id == 1 else 'NEGATIVE'
    return label, score

In [ ]:
news[['sentiment_label', 'sentiment_score']] = news.apply(lambda x: pd.Series(get_sentiment(x['input_ids'], x['attention_mask'])), axis=1)

news

### Text Embedding

In [ ]:
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def get_embeddings_from_dataframe(df, model, batch_size=32):
    num_rows = len(df)
    embeddings = []

    input_ids_list = df['input_ids'].tolist()
    attention_mask_list = df['attention_mask'].tolist()

    for start in tqdm(range(0, num_rows, batch_size)):
        end = min(start + batch_size, num_rows)
        batch_input_ids = input_ids_list[start:end]
        batch_attention_mask = attention_mask_list[start:end]

        input_ids = torch.tensor(batch_input_ids).to(device)
        attention_mask = torch.tensor(batch_attention_mask).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_emb = outputs.pooler_output.cpu().numpy()

        embeddings.append(batch_emb)

    return np.vstack(embeddings)


In [ ]:
embeddings = get_embeddings_from_dataframe(news, model)
news['embedding'] = list(embeddings)

news

### Normalize Data

In [ ]:
def mean_sentiment_score(row, sentiment):
    scores = row['sentiment_score']
    labels = row['sentiment_label']
    total = 0
    count = 0
    for label, score in zip(labels, scores):
        if label == sentiment:
            total += score
            count += 1
    return total / count if count > 0 else 0.0

In [ ]:
grouped_news = news.groupby('date', as_index=False).agg({
    'title': list,             
    'content': list,
    'title_clean': list,
    'content_clean': list,
    'title_abbreviation': list,
    'content_abbreviation': list,
    'title_entity': list,
    'content_entity': list,
    'title_stopword_removed': list,
    'content_stopword_removed': list,
    'title_stemmed': list,
    'content_stemmed': list,
    'full_text': list,
    'input_ids': list,     
    'attention_mask': list,
    'sentiment_label': list,
    'sentiment_score': list,
    'embedding': list          
})
grouped_news['emb_mean'] = grouped_news['embedding'].apply(lambda emb_list: np.mean(np.stack(emb_list), axis=0) if len(emb_list) > 0 else None)
grouped_news['positive_grouped_news_sentiment'] = grouped_news['sentiment_label'].apply(lambda labels: labels.count('POSITIVE') if labels else 0)
grouped_news['negative_grouped_news_sentiment'] = grouped_news['sentiment_label'].apply(lambda labels: labels.count('NEGATIVE') if labels else 0)
grouped_news['mean_sentiment_score_positive'] = grouped_news.apply(lambda new: mean_sentiment_score(new, 'POSITIVE'), axis=1)
grouped_news['mean_sentiment_score_negative'] = grouped_news.apply(lambda new: mean_sentiment_score(new, 'NEGATIVE'), axis=1)

grouped_news

In [ ]:
news = grouped_news
news['date'] = pd.to_datetime(news['date']).astype('datetime64[s]')
news

In [ ]:
X_emb = np.vstack(news['emb_mean'].values)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_emb)
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_scaled)

for i in range(50):
    news[f'sentiment_pca_{i}'] = X_pca[:, i]

news

In [ ]:
news.to_csv(f'data/Processed-News/{kode_saham}.csv', index=False)

### Data Preprocessing

In [ ]:
processed_news = pd.read_csv(f'data/Processed-News/{kode_saham}.csv')
processed_news.select_dtypes('number').fillna(0)
processed_news

In [ ]:
processed_news['date'] = pd.to_datetime(processed_news['date']).astype('datetime64[s]')
df['Date'] = pd.to_datetime(df['Date']).astype('datetime64[s]')
ihsg['Date'] = pd.to_datetime(ihsg['Date']).astype('datetime64[s]')
ihsg['Close_ihsg'] = ihsg['Close']
orderbook['date'] = pd.to_datetime(orderbook['date']).astype('datetime64[s]')

processed_news.rename(columns={'date': 'Date'}, inplace=True)
df = pd.merge(df, processed_news, on='Date', how='left')
df = pd.merge(df, ihsg[['Date', 'Close_ihsg']], on='Date', how='left')
df = pd.merge(df, orderbook[['Date', 'Value', 'Offer_Value', 'Offer_Volume', 'Bid_Value', 'Bid_Volume', 'Foreign_Sell', 'Foreign_Buy']], on='Date', how='left')
df.select_dtypes('number').fillna(0, inplace=True)

df


In [ ]:
df = df[['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Close_ihsg', 'Value', 'Offer_Value', 'Offer_Volume', 'Bid_Value', 'Bid_Volume', 'Foreign_Sell', 'Foreign_Buy', 'positive_grouped_news_sentiment', 'negative_grouped_news_sentiment', 'mean_sentiment_score_positive', 'mean_sentiment_score_negative']+[f'sentiment_pca_{i}' for i in range(50)]]

df.columns

In [ ]:
string_cols = df.select_dtypes(include=['object']).columns
string_cols

In [ ]:
num_cols = df.select_dtypes(include=['number']).columns
string_cols = df.select_dtypes(include=['object']).columns
df[num_cols] = df[num_cols].fillna(0)
df[string_cols] = df[string_cols].fillna('')

In [ ]:
df.info()

# Addons Data

## RSI

Indikator ini merupakan indikator momentum yang dapat menjadi tolak ukur investor menentukan titik beli dan titik jual suatu saham [5]. Indikator ini di normalisasi dari skala 1 – 100 dengan area dibawah 30 merupakan area jenuh jual (oversold) dan area diatas 70 adalah area jenuh beli (overbought) [6]. Persamaan yang digunakan dalam menghitung RSI suatu periode saham adalah 

$$ RSI = 100 - \frac{100}{1 + RS} $$
$$ RS = \frac{SMA(U)}{SMA(D)}$$

RSI: Relative Strength Index

RS: Relative Strength

SMA(U): Simple Moving Average Up (Jika harga naik, U = Change, D = 0)

SMA(D): Simple Moving Average Down (Jika harga turun, U = 0, D = Change). Jika harga tetap, U = 0, D = 0.

Indikator ini kemudian dikembangkan dengan membandingkan harga saat ini dengan rata-rata nya yang disebut dengan Stochastik RSI (Stoch). 

$$Stoch(K\%) = \frac{(RSI - min(RSI))}{(max(RSI) - min(RSI))} \times 100 $$
$$D\% = SMA(K\%)$$

Dengan:

$K\%$ = Normalisasi RSI ke skala 0-1 dikali 100

$D\%$ = Rata - rata K% selama 3 hari

In [ ]:
def compute_rsi(series, period = 14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = delta.where(delta < 0, 0.0)

    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (100 + rs))
    return rsi

In [ ]:
def compute_stoch_rsi(rsi, lookback = 14):
    min_rsi = rsi.rolling(window=lookback).min()
    max_rsi = rsi.rolling(window=lookback).max()

    range_rsi = max_rsi - min_rsi
    k = ((rsi - min_rsi) / range_rsi) * 100
    d = k.rolling(window = 3).mean()
    
    return k, d
    

## MACD (Moving Average Convergence/Divergence)

merupakan indikator trend (Trend Indicator) yang menampilkan perbedaan antara pergerakan harga exponential rata-rata (Exponential  Moving Average (EMA)) yang lebih cepat atau lebih lambat dari harga penutupan (Close Price) [5]. Indikator ini dirancang untuk mengidentifikasi perubahan yang terjadi pada trend, namun tidak direkomendasikan untuk digunakan untuk pasar dengan kondisi yang volatile. Terdapat 3 signal yang diberikan dari indikator MACD ini, yaitu [5]:

1.	MACD menyilang dengan garis sinyal (Signal Line)
2.	MACD menyilang dengan garis 0
3.	Perbedaan antara harga dan level MACD. 

3 signal dari indikator MACD diatas merupakan sinyal beli jika MACD menyilang ke atas terhadap signal line dan akan menjadi sinyal jual jika MACD menyilang ke bawah terhadap signal line [5]. Histogram pada MACD juga dapat menjadi petunjuk jika persilangan akan segera terjadi [5] dan menjadi tanda kekuatan sinyal yang diberikan, yaitu jika menyilang ke atas dibawah histogram menandakan sinyal beli lemah, jika menyilang ke atas diatas histogram menandakan sinyal beli kuat, jika menyilang kebawah diatas histogram menandakan sinyal jual lemah, jika menyilang kebawah dibawah histogram menandakan sinyal jual kuat [6]. 

$$MACD_{line} = EMA(12)_{t} - EMA(26)_{t}$$
$$Signal_{line} = EMA(9)_{MACD_{line}}$$
$$Histogram = MACD_{line} - Signal_{line}$$


In [ ]:
def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    macd_signal = macd_line.ewm(span=signal, adjust=False).mean()
    macd_hist = macd_line - macd_signal
    
    return macd_line, macd_signal, macd_hist

## Moving Average

Moving Averages (MA) merupakan indikator teknikal yang mencerminkan harga rata-rata dari nilai pergerakan suatu saham di dalam waktu tertentu dengan rentang waktu sesuai dengan yang diatur oleh investor [4]. Terdapat 3 jenis MA, yaitu Simple Moving Averages (SMA), Weighted Moving Average (WMA), dan Exponential Moving Average (EMA) [6].

Perbedaan dari ketiga jenis MA tersebut [6], SMA merupakan rata-rata dari nilai pergerakan suatu saham di waktu tertentu dan rentang waktu tertentu secara sederhana dengan kekurangannya yang menganggap semua periode sebelumnya sama pentingnya dalam pembentukan trend, walaupun angka penutupan lebih efektif dalam membentuk pergerakan harga. WMA merupakan MA yang dirancang untuk menutupi kekurangan dari SMA dengan harga pada suatu periode diberi bobot sesuai dengan urutan seri yang membuatnya memiliki bobot yang lebih besar diberikan pada pergerakan harga terbaru, namun hal ini masih menimbulkan kekurangan berupa noise karena bobot yang diterapkan pada keseluruhan data. EMA dirancang untuk mengurangi noise dari WMA dan mengekspresikan pergerakan harga terbaru dengan lebih kuat. Hal ini dikarenakan EMA dihitung secara rekursif berdasarkan periode sebelumnya yang memberikan penghalusan yang baik dan mengurangi banyak noise.

$$SMA(n)_{t} = \frac{\sum_{i=1}^{n} A_{i}}{n} = \overline{x}n_{t} = EMA(n)_{1}$$
$$WMA(n)_{t} = \frac{\sum_{i=1}^{n} A_{i}(n - i)}{\frac{n(n+1)}{2}}$$
$$EMA(n)_{t} = A_{t}(\frac{2}{n+1}) + EMA(n)_{t-1}(1 - \frac{2}{n+1})$$

dengan

$SMA(n)_{t}$ = Simple moving average

$WMA(n)_{t}$ = Weighted Moving Average

$EMA(n)_{t}$ = Exponential Moving Average

$\overline{x}n_{t}$ = rata rata harga saham pada waktu ke t

$A_{i}$, $A_{t}$ = harga saham pada periode ke-i atau periode ke-t

In [ ]:
def compute_ma(series, n):
    sma = series.rolling(window = n).mean()
    
    wma = series.rolling(window = n).apply(lambda window: sum(window[i] * (n - (i + 1)) for i in range(n)), raw=True) 
    wma = wma / ((n * (n + 1))/2)

    alpha  = 2 / (n + 1)
    ema = series.copy().astype(float)
    ema.iloc[0] = series.iloc[0]
    for i in range(1, len(series)):
        ema.iloc[i] = series.iloc[i] * alpha + ema.iloc[i-1] * (1 - alpha)
    
    return sma, wma, ema

## Boolinger Bands

Indikator ini merupakan salah satu indikator teknikal berguna dalam mendapatkan informasi apakah pasar memiliki volatilitas yang tinggi atau rendah (Volatility Indicator) [5]. Indikator ini melibatkan perhitungan volatilitas harga dan bekerja dengan dua garis batas, yaitu batas atas dan batas bawah serta garis batas tengah mengikuti rata-rata pergerakan harga sepanjang periode tertentu. Batas atas disebut dengan upperband, batas bawah disebut dengan lowerband, dan garis tengah disebut dengan middle band [4]. Perlu diketahui, nilai volatilitas yang berada dalam interval positif, yaitu dari 0 hingga $\infty (0\ \le\ \sigma\le\ \infty)$ menunjukkan harga saham berubah, bila rendah, dapat dikatakan konstan. Bila \sigma^2 tidak diketahui, grafik batas BB dari SMA periode ke-n pada waktu ke-t adalah:

$$\sigma_{B(n)_{t}} = \sqrt{\frac{\sum_{i=1}^{n}(X_{i} - \overline{X}(n)_{t})^{2}}{n}}$$
$$LB(n)_{t} = \overline{X}(n)_{t} - k\sigma_{B(n)_{t}}$$
$$MB(n)_{t} = \overline{X}(n)_{t}$$
$$UB(n)_{t} = \overline{X}(n)_{t} + k\sigma_{B(n)_{t}}$$

dengan

$\sigma_{B(n)_{t}}$ = Volatilitas Harga

$LB(n)_{t}$ = Lowerband

$MB(n)_{t}$ = Middleband

$UB(n)_{t}$ = Upperband

$n$ = banyak data harga saham (standar yang disarankan bernilai 20)

$X$ = Harga saham

$k$ = ketentuan nilai koefisien (standar yang disarankan 2)

$\overline{X}(n)_{t}$ = SMA periode ke t

Percent Bollinger Band (PBB) adalah indikator yang didapatkan dari pergerakan garis elastis upperband dan lowerband. Apabila PBB lebih dari 1, pergerakan harga berada di atas garis elastis upperband. Sebaliknya, jika PBB kurang dari 0, pergerakan harga berada dibawah garis lower band. Disaat nilai PBB adalah 0.5, pergerakan harga tepat berada dalam middle band [4]. Persamaan yang digunakan adalah: 

$$PBB(n)_{t} = \frac{X_{t} - LB(n)_{t}}{UB(n)_{t} - LB(n)_{t}}$$

dengan: 
$PBB(n)_{t}$ = Percent Bolinger Bands pada harga saham ke n

In [ ]:
def compute_bolinger_band(series, n = 20, k = 2):
    mb = series.rolling(window=n).mean()
    volatile = np.sqrt((((series - mb) ** 2)).rolling(window = n).mean())
    lb = mb - (k * volatile)
    ub = mb + (k * volatile)
    pbb = (series - lb) / (ub - lb)

    return lb, mb, ub, pbb

## Return

Indikator teknikal berikut digunakan untuk mengetahui berapakah hasil yang didapatkan ketika berinvestasi berdasarkan capital gain atau capital loss dari suatu index saham [4]. Persamaan yang digunakan adalah

$$R(n)_{t} = ln(\frac{X(n)}{X(n-t)})$$

dengan $X(n)$ adalah data saham pada periode ke-n

In [ ]:
def compute_return(series, t = 1):
    result = [series.iloc[i] / series.iloc[i - t] for i in range(len(series))]
    result = np.log(result) * 100
    return result

def compute_difference(series, t = 1):
    result = series - series.shift(t)
    return result


## Parabolic Stop and Reversal (PSAR)

Indikator yang digunakan untuk menangkap harga saat ini dan perubahan trend [6]. Jika harga < PSAR berarti sinyal beli. Jika harga > PSAR berarti sinyal jual.

$$PSAR = C + AF(EPT - C)$$

dengan
$PSAR$ = Parabolic stop and reversal

$C$ = Nilai $PSAR$ periode sebelumnya

$AF$ = Acceleration Factor (percepatan agar $SAR$ mendekati harga saat trend semakin kuat (awal: 0.02, $EP$ Baru -> +0.02, Max: 0.2, Reset ketika harga < $PSAR$))

$EPT$ = Extreme Point (titik harga paling ekstrem (highest high: uptrend, lowest low: downtrend))

In [ ]:
def compute_psar(data, af_start = 0.02, af_step = 0.02, af_max = 0.2):
    psar  = pd.Series(index=data['High'].index, dtype=float)
    trend = 1
    af = af_start
    ep = None
    psar.iloc[1] = np.nan

    if data['High'].iloc[1] > data['High'].iloc[0]:
        trend = 1
        ep = data['High'].iloc[1]
        psar.iloc[1] = data['Low'].iloc[0]
    else:
        trend = -1
        ep = data['Low'].iloc[1]
        psar.iloc[1] = data['High'].iloc[0]
    
    for i in range(2, len(data['High'])):
        sar_prev = psar.iloc[i-1]
        if trend == 1:
            psar.iloc[i] = sar_prev + (af * (ep - sar_prev))
            psar.iloc[i] = min(psar.iloc[i], data['Low'].iloc[i-1], data['Low'].iloc[i-2] if i>1 else data['Low'].iloc[i-1])
            if psar.iloc[i] > data['Low'].iloc[i]:
                trend = -1
                psar.iloc[i] = ep
                af = af_start
                ep = data['Low'].iloc[i]
                sar_prev = psar.iloc[i]
                if trend == -1:
                    psar.iloc[i] = sar_prev + (af * (ep - sar_prev))
                    psar.iloc[i] = max(psar.iloc[i], data['High'].iloc[i - 1], data['High'].iloc[i - 2] if i > 1 else data['High'].iloc[i - 1])
                    continue
        else:
            psar.iloc[i] = sar_prev + (af * (ep - sar_prev))
            psar.iloc[i] = max(psar.iloc[i], data['High'].iloc[i-1], data['High'].iloc[i-2] if i > 1 else data['High'].iloc[i-1])
            if psar.iloc[i] < data['High'].iloc[i]:
                trend = 1
                psar.iloc[i] = ep
                af = af_start
                ep = data['High'].iloc[i]
                sar_prev = psar.iloc[i]
                if trend == 1:
                    psar.iloc[i] = sar_prev + (af * (ep - sar_prev))
                    psar.iloc[i] = min(psar.iloc[i], data['Low'].iloc[i-1], data['Low'].iloc[i-2] if i < 1 else data['Low'].iloc[i-1])
                    continue
            
        if trend == 1:
            if data['High'].iloc[i] > ep:
                ep = data['High'].iloc[i]
                af = min(af + af_step, af_max)
        else:
            if data['Low'].iloc[i] < ep:
                ep = data['Low'].iloc[i]
                af = min(af + af_step, af_max)
    
    return psar


## Data Adjustment

In [ ]:
# MA
df['SMA5'], df['WMA5'], df['EMA5'] = compute_ma(df['Close'], 5)
df['SMA20'], df['WMA20'], df['EMA20'] = compute_ma(df['Close'], 20)
df['SMA50'], df['WMA50'], df['EMA50'] = compute_ma(df['Close'], 50)
df['SMA100'], df['WMA100'], df['EMA100'] = compute_ma(df['Close'], 100)

# Bolinger band
df['lb_bolinger'], df['mb_bolinger'], df['ub_bolinger'], df['percent_bolinger'] = compute_bolinger_band(df['Close'])

# return harian
df['Return'] = compute_return(df['Close'])
df['Return_Num'] = compute_difference(df['Close'])

# MACD
df['MACD_Line'], df['MACD_Signal'], df['MACD_Hist'] = compute_macd(df['Close'])

# RSI & Stoch RSI
df['RSI'] = compute_rsi(df['Close'])
df['Stoch_RSI_k%'], df['Stoch_RSI_d%'] = compute_stoch_rsi(df['RSI'])

# Parabolic Stop and Reversal
df['PSAR'] = compute_psar(df)

# IHSG Return
df['return_ihsg'] = compute_return(df['Close_ihsg'])
df

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)
df.dropna(inplace=True)
df

In [ ]:
df.columns

In [ ]:
df.describe()

# Handling Outliers

In [ ]:
selected_columns = ['Return_Num'] 

plt.figure(figsize=(12, 4))
for i, col in enumerate(selected_columns, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(f'Boxplot - {col}')
    plt.ylabel(col)
plt.tight_layout()
plt.show()

for col in selected_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    plt.figure(figsize=(10, 4))
    plt.hist(df[col], bins=50, alpha=0.7, label=col)
    plt.axvline(lower_bound, color='red', linestyle='--', label=f'Batas bawah IQR ({lower_bound:.2f})')
    plt.axvline(upper_bound, color='red', linestyle='--', label=f'Batas atas IQR ({upper_bound:.2f})')
    plt.title(f'Histogram {col} dengan batas outlier IQR')
    plt.legend()
    plt.show()

plt.figure(figsize=(14, 6))
for i, col in enumerate(selected_columns, 1):
    plt.subplot(1, 3, i)
    plt.scatter(df.index, df[col], s=1, alpha=0.5, color='blue')
    plt.title(f'{col} vs Waktu')
    plt.xlabel('Indeks baris')
    plt.ylabel(col)
plt.tight_layout()
plt.show()

for col in selected_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    plt.figure(figsize=(12, 4))
    plt.plot(df.index, df[col], 'b.', markersize=2, label='Data normal')
    plt.plot(outliers.index, outliers[col], 'r.', markersize=5, label='Outlier')
    plt.axhline(y=lower, color='orange', linestyle='--', label='Batas IQR bawah')
    plt.axhline(y=upper, color='orange', linestyle='--', label='Batas IQR atas')
    plt.title(f'Outlier pada {col} (IQR dengan multiplier=1.5)')
    plt.xlabel('Indeks baris')
    plt.ylabel(col)
    plt.legend()
    plt.show()
    
    print(f"Jumlah outlier pada {col}: {len(outliers)} ({100*len(outliers)/len(df):.2f}% dari data)")

- Volume dan Close: Jelas tidak memiliki outlier (semua titik berada dalam rentang whisker).

- Return: Nilai minimum -0.10 (-10%) secara matematis berada di bawah batas bawah IQR (-0.05), tetapi pada boxplot terlihat whisker memanjang hingga -0.10. Itu berarti plot yang Anda gunakan mungkin menggunakan whisker yang berbeda (misalnya persentil 5-95, atau tidak menggunakan IQR sama sekali). Atau bisa juga data memang memiliki outlier tipis di ekor kiri, tetapi jumlahnya sangat sedikit.

# Distribution Data

In [ ]:
num_vars = df.shape[1]
n_cols = 5
n_rows = -(-num_vars // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, column in enumerate(df.select_dtypes(include=['number']).columns):
    df[column].hist(ax=axes[i], bins=20, edgecolor='black')
    axes[i].set_title(column)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## Data Correlation

In [ ]:
for i in ['Return', 'Return_Num']:
    target_corr = df.select_dtypes(include=['number']).corr()[i]
    target_corr_sorted = target_corr.abs().sort_values(ascending=False)

    plt.figure(figsize=(35, 20))
    target_corr_sorted.plot(kind='bar')
    plt.title(f'Correlation with {i}')
    plt.xlabel('Variabels')
    plt.ylabel('Correlation Coefficient')
plt.show()

## Feature Engineering

### Non Linear Auto Regression feature

In [ ]:
features = df.columns.drop(['Date', 'Return', 'Return_Num'])

X = df[features]

y = df['Return']

In [ ]:
X

In [ ]:
y

Hitung korelasi Pearson antara setiap fitur dan target. Pilih fitur yang nilai absolut korelasinya > 0.1.

- Mengapa threshold 0.3? Karena di bawah itu, hubungan linear sangat lemah fitur seperti Volume_Change biasanya di bawah 0.1 dan cenderung hanya menambah noise.

- Catatan: Korelasi linear tidak menangkap hubungan non-linear, tetapi untuk langkah awal penyaringan cukup baik. Model CNN-LSTM nanti bisa menangkap pola non-linear dari fitur yang tersisa.

# Data Splitting

In [ ]:
split_index = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

## Data Normalization

In [ ]:
# scaler_X = MinMaxScaler()

# X_scaled = scaler_X.fit_transform(X)

# Data Training

In [ ]:
model = Sequential([
    
])

model.fit(X_train, y_train)

# Data Predict

In [ ]:
pred = model.predict(X_test)

# Data Evaluation

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE: ", rmse) 

# Trading Plan

In [ ]:
signal = []

for p in pred:
    if p >= 0.01:
        signal.append("BUY")
    elif p <= -0.01:
        signal.append("SELL")
    else:
        signal.append("HOLD")

In [ ]:
result = pd.DataFrame({
    "pred_return": pred,
    "signal": signal,
    "price": X_test['Close']
})

result

# Strategy Simulation

In [ ]:
data = result.copy()
strategy_return = []

for i in range(len(data)):
    actual_return = data["price"].shift(-1) / data["price"] - 1
    r = actual_return.iloc[i]
    s = data["signal"].iloc[i]

    if s == "BUY":
        strategy_return.append(r)
    elif s == "SELL":
        strategy_return.append(-r)
    else: 
        strategy_return.append(0)
    
data['strategy_return'] = strategy_return
data

In [ ]:
position = 0
entry_price = 0
equity = 100
equity_curve = []

for i in range(len(data)):
    price = data["price"].iloc[i]
    signal = data["signal"].iloc[i]

    if signal == "BUY" and position == 0:
        position = 1
        entry_price = price

    elif signal == "SELL" and position == 1:
        pnl = (price - entry_price) / entry_price
        equity *= (1 + pnl)
        position = 0
        entry_price = 0

    equity_curve.append(equity)

data["strategy_equity"] = equity_curve
data

In [ ]:
data["market_return"] = data["price"].pct_change().shift(-1)
data["market_equity"] = (1 + data["market_return"].fillna(0)).cumprod() * (100)
buy = data[data["signal"] == "BUY"]
sell = data[data["signal"] == "SELL"]

data.fillna(0, inplace=True)
data

In [ ]:
plt.figure(figsize=(20,6))

plt.plot(data["price"], label="Close Price", alpha=0.7)
plt.plot(data["strategy_equity"], label="Equity", alpha=0.7)

plt.scatter(buy.index, buy["price"], label="BUY", marker="^", color="green")
plt.scatter(sell.index, sell["price"], label="SELL", marker="v", color="red")

plt.title("Price with Buy/Sell Signals")
plt.legend()
plt.show()

In [ ]:
data["strategy_pnl"] = data["strategy_return"]

print("Total return strategy:", data["strategy_equity"].iloc[-1])
print("Total return market:", data["market_equity"].iloc[-1])
print("Win rate BUY signal:", (data["strategy_return"] > 0).mean())

In [ ]:
import joblib
from tensorflow.keras.models import save_model

# --- 1. Simpan Model LSTM ---
# Gunakan format .keras untuk menyimpan arsitektur, bobot, dan konfigurasi training.
model.save('model_saham_lstm.keras')
print("Model LSTM berhasil disimpan.")

# --- 2. Simpan Scaler dan Fitur Lainnya dengan Joblib ---
# Scaler untuk fitur input (X)
joblib.dump(scaler_X, 'scaler_X.joblib')
# Scaler untuk target (y)
joblib.dump(scaler_y, 'scaler_y.joblib')
# Daftar nama fitur yang terpilih
joblib.dump(selected_features, 'selected_features.joblib')

print("Scaler dan konfigurasi berhasil disimpan.")